# 12-4 검색 증강 생성으로 환각 줄이기

본 노트북은 12-4절 본문 예제 [코드 12-18]~[코드 12-22]를 모아 위에서 아래로
실행한다. 문장 임베딩 모델로 문서를 인덱싱하고, 코사인 유사도로 관련 문서를
검색해 LLM에 함께 전달하는 간단한 RAG 시스템(모델 18: 진실된 인공지능 대화
서비스)을 만든다.

**API 키 안내** - 생성 단계는 기본적으로 Groq API를 사용한다. Groq 콘솔
(https://console.groq.com)에서 무료 API 키를 발급해 `GROQ_API_KEY`에 넣으면
실제 API를 호출한다. 키가 없으면 같은 인터페이스를 그대로 유지한 채 로컬 한국어
LLM(Bllossom-3B)으로 생성 단계를 대체하므로, 키 없이도(GPU 환경) 실제 RAG
결과를 확인할 수 있다. 다만 로컬 3B 모델은 Groq의 8B 모델보다 품질이 낮아 본문
[표 12-12]의 결과와 답변이 다를 수 있다.

**이 장의 코드 컨벤션 알림** - 12장은 허깅페이스 라이브러리를 중심 주제로
다루므로, `MODEL_NAME`, `GROQ_MODEL` 등 상수 표기는 공통 컨벤션을 따르지만
`model`, `embed_model`, `groq_client` 같은 객체명은 통용 표기를 그대로 사용한다.

다루는 내용
- 본문 비공개 항목: 예제용 근거 문서 6편(documents), Groq 키가 없을 때의 로컬 LLM 폴백
- [코드 12-18] 문서 임베딩 생성
- [코드 12-19] 질문과 가장 유사한 문서 검색
- [코드 12-20] Groq 클라이언트 생성과 호출
- [코드 12-21] RAG 프롬프트 구성과 생성 함수
- [코드 12-22] 통합 RAG 파이프라인 + RAG/비-RAG 답변 비교 ([표 12-12] 대응)

> 사용 모델: snunlp/KR-SBERT-V40K-klueNLI-augSTS (MIT License),
> Groq llama-3.1-8b-instant (Llama 3.1 커뮤니티 라이선스),
> 로컬 폴백 Bllossom/llama-3.2-Korean-Bllossom-3B (Llama 3.2 커뮤니티 라이선스).

In [ ]:
# 참고 - 라이브러리 설치 (이미 설치된 경우 건너뛰어도 좋다)
# !pip install -q groq sentence-transformers transformers

In [2]:
# 참고 - 공통 라이브러리 경로 설정
import sys
sys.path.append('../../')

from code_reference import common

device = common.get_device()


CUDA를 사용합니다.


In [ ]:
# 참고 - Groq API 키 설정
# 발급받은 키가 있으면 아래 변수에 직접 넣거나, 셸에서 환경 변수로 설정한다.
#   리눅스/macOS:  export GROQ_API_KEY='발급받은 키'
#   주피터 노트북: os.environ['GROQ_API_KEY'] = '발급받은 키'
# 키가 비어 있으면 이 노트북은 로컬 한국어 LLM(Bllossom-3B)으로 생성 단계를 대체한다.
import os

GROQ_API_KEY = os.environ.get('GROQ_API_KEY', '')  # 'groq_key' 자리에 키 입력
USE_LOCAL_LLM = not bool(GROQ_API_KEY)
print(f'로컬 LLM 폴백 모드: {USE_LOCAL_LLM}')

In [ ]:
# 참고 - 라이브러리 import 와 시드 고정
SEED = 42
common.set_seed(SEED)
device = common.get_device()

# # 참고 - 라이브러리 import 와 시드 고정
# import random
# import warnings

# import numpy as np
# import torch
# import torch.nn.functional as F
# from sentence_transformers import SentenceTransformer

# warnings.filterwarnings('ignore')

# SEED = 42
# random.seed(SEED)
# np.random.seed(SEED)
# torch.manual_seed(SEED)
# if torch.cuda.is_available():
#     torch.cuda.manual_seed_all(SEED)


## 참고 - 예제용 근거 문서 집합 정의

답변 근거가 될 LLM·트랜스포머 관련 짧은 글 6편을 딕셔너리(`documents`)로 둔다.
실제 서비스라면 수집·청킹 단계를 거치지만, 예제는 이를 생략하고 짧은 문서를
직접 코드 안에 둔다.


In [6]:
# 참고 - LLM·트랜스포머 관련 짧은 글 6편
documents = [
    {'title': '트랜스포머 아키텍처',
     'content': (
         '트랜스포머는 2017년 구글이 발표한 신경망 구조로, 어텐션 메커니즘만으로 '
         '순차 데이터를 처리한다. 인코더와 디코더가 모두 셀프 어텐션과 피드포워드 '
         '계층의 반복으로 구성되며, 위치 정보는 위치 인코딩을 통해 더한다. 이후 '
         '거의 모든 대규모 언어 모델의 기본 구조가 되었다.')},
    {'title': 'GPT 시리즈',
     'content': (
         'GPT는 OpenAI가 공개한 디코더 전용 트랜스포머 계열의 언어 모델 시리즈로, '
         'GPT-3(2020)에서 1,750억 파라미터로 규모를 크게 키워 화제가 됐다. GPT-4는 '
         '멀티모달 입력을 지원하며 ChatGPT 서비스의 기반 모델로 사용된다. 대규모 '
         '사전 학습 후 지시어 미세 조정과 RLHF로 정렬되는 파이프라인을 따른다.')},
    {'title': 'LLaMA',
     'content': (
         'LLaMA는 Meta가 2023년부터 공개한 오픈 가중치 대규모 언어 모델 시리즈다. '
         'LLaMA 2와 LLaMA 3로 이어지며 연구·상용 모두 사용 가능한 라이선스로 배포돼 '
         '오픈소스 LLM 생태계의 표준이 됐다. 한국어 특화 파생 모델인 Bllossom도 '
         'LLaMA 3 계열을 기반으로 한다.')},
    {'title': 'RAG(검색 증강 생성)',
     'content': (
         'RAG는 2020년 메타(Meta) AI 연구팀이 발표한 기법으로, LLM의 환각과 학습 후 '
         '정보 부재 문제를 외부 문서 검색으로 보완한다. 파이프라인은 (1) 문서를 임베딩 '
         '벡터로 변환해 저장, (2) 질문을 임베딩해 유사한 문서를 검색, (3) 검색된 문서를 '
         '컨텍스트로 LLM에 전달해 답변을 생성하는 세 단계로 구성된다.')},
    {'title': 'Groq',
     'content': (
         'Groq는 LPU(Language Processing Unit)라는 AI 추론 전용 칩을 개발한 '
         '스타트업이다. 오픈소스 LLM의 API 서비스를 함께 제공하며, 호출 인터페이스가 '
         'OpenAI API와 동일해 코드 호환성이 높다. 무료 티어에서도 학습·실험용으로 '
         '충분한 사용량을 제공해 RAG 같은 빠른 응답이 필요한 시스템에 적합하다.')},
    {'title': '파인튜닝과 프롬프트 엔지니어링',
     'content': (
         '파인튜닝은 사전 학습 모델을 작업 데이터로 추가 학습해 모델의 동작 자체를 '
         '바꾸는 방법이다. 반면 프롬프트 엔지니어링은 모델은 그대로 두고 입력 프롬프트를 '
         '잘 구성해 원하는 출력을 끌어내는 방법이다. RAG는 후자의 발전된 형태로, 모델은 '
         '그대로 두고 외부 문서를 동적으로 컨텍스트에 끼워 넣어 답변을 보강한다.')},
]
print(f'문서 개수: {len(documents)}')
for doc in documents:
    print(f"- {doc['title']} ({len(doc['content'])}자)")


문서 개수: 6
- 트랜스포머 아키텍처 (152자)
- GPT 시리즈 (183자)
- LLaMA (163자)
- RAG(검색 증강 생성) (182자)
- Groq (187자)
- 파인튜닝과 프롬프트 엔지니어링 (174자)


## [코드 12-18] 문서 임베딩 생성

한국어 문장 유사도에 특화된 KR-SBERT-KLUE 모델을 `SentenceTransformer`로
불러와 문서 임베딩을 만든다. 코사인 유사도를 내적으로 계산하기 위해 L2 정규화해
둔다.


In [ ]:
###############################################################################
# 코드 12-18 - 문서 임베딩 생성
###############################################################################
embed_model = SentenceTransformer('snunlp/KR-SBERT-V40K-klueNLI-augSTS')


def build_index(documents, model):
    texts = [doc['content'] for doc in documents]
    embeddings = model.encode(texts, convert_to_tensor=True)
    # L2 정규화: 코사인 유사도를 내적으로 계산하기 위한 전처리
    embeddings = F.normalize(embeddings, p=2, dim=1)
    return embeddings


doc_embeddings = build_index(documents, embed_model)
print(f'임베딩 shape: {doc_embeddings.shape}')   # (문서 수, 임베딩 차원)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

임베딩 shape: torch.Size([6, 768])


## [코드 12-19] 질문과 가장 유사한 문서 검색

질문도 같은 방식으로 임베딩·정규화한 뒤, 정규화된 문서 임베딩과의 행렬곱 한
번으로 모든 문서의 코사인 유사도를 구한다. `topk`로 상위 문서를 고른다.


In [8]:
# 코드 12-19 - 질문과 가장 유사한 문서 검색
def retrieve(query, documents, doc_embeddings, embed_model, top_k=2):
    # doc_embeddings는 build_index()에서 이미 L2 정규화된 상태로 전달받는다고 가정.
    # 질문도 같은 방식으로 임베딩하고 정규화한다.
    query_embedding = embed_model.encode(query, convert_to_tensor=True)
    # 1차원 벡터이므로 dim=0 (배치 텐서면 dim=1 또는 차원과 무관하게 dim=-1)
    query_embedding = F.normalize(query_embedding, p=2, dim=0)
    # 행렬 곱으로 코사인 유사도 계산 (정규화된 벡터의 내적 = 코사인 유사도)
    scores = torch.matmul(doc_embeddings, query_embedding)
    top_indices = torch.topk(scores, k=top_k).indices.tolist()
    return ([documents[i] for i in top_indices],
            [scores[i].item() for i in top_indices])


retrieved_docs, scores = retrieve(
    '트랜스포머 모델은 어떤 구조인가요?', documents, doc_embeddings, embed_model,
)
for doc, score in zip(retrieved_docs, scores):
    print(f'[유사도: {score:.4f}] {doc["title"]}')


[유사도: 0.3957] 트랜스포머 아키텍처
[유사도: 0.3523] GPT 시리즈


## 참고 - 생성 클라이언트 정의 (Groq 또는 로컬 LLM)

`GROQ_API_KEY`가 있으면 실제 `groq.Groq` 클라이언트를 사용한다. 키가 없으면 로컬
한국어 LLM(Bllossom-3B)을 불러와 `chat.completions.create()`와 동일한 인터페이스를
제공하는 드롭인 클라이언트를 만든다. 덕분에 이후 [코드 12-20]~[코드 12-22]는 생성
백엔드와 무관하게 그대로 동작한다(코드 12-22의 `generate_fn` 추상화와 같은 맥락).

In [ ]:
# 참고 - 생성 클라이언트 (실제 Groq API 또는 로컬 LLM 폴백)
GROQ_MODEL = 'llama-3.1-8b-instant'
MAX_GEN_TOKEN = 512

if not USE_LOCAL_LLM:
    from groq import Groq
    groq_client = Groq(api_key=GROQ_API_KEY)
else:
    # Groq API 키가 없을 때: 로컬 한국어 LLM(Bllossom-3B)으로 생성 단계를 대체한다.
    # groq_client.chat.completions.create(...)와 동일한 인터페이스를 제공하는
    # 드롭인 클라이언트라, 이후 코드는 백엔드와 무관하게 그대로 동작한다.
    from types import SimpleNamespace

    from transformers import AutoModelForCausalLM, AutoTokenizer

    LOCAL_MODEL_NAME = 'Bllossom/llama-3.2-Korean-Bllossom-3B'
    print(f'Groq API 키가 없어 로컬 LLM({LOCAL_MODEL_NAME})으로 생성합니다. '
          '최초 1회 모델 다운로드와 로딩에 시간이 걸릴 수 있습니다.')

    _local_tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_NAME)
    _local_model = AutoModelForCausalLM.from_pretrained(
        LOCAL_MODEL_NAME, torch_dtype=torch.float16,
    ).to(device)
    _local_model.eval()
    # VRAM이 빠듯하면(8GB 미만) [코드 12-14]의 BitsAndBytesConfig로 4비트 로드도 가능

    # Llama 3 계열 종료 토큰: 문장 끝 토큰과 <|eot_id|>를 함께 종료 신호로 사용
    _eot_id = _local_tokenizer.convert_tokens_to_ids('<|eot_id|>')
    _terminators = list({t for t in (_local_tokenizer.eos_token_id, _eot_id)
                         if isinstance(t, int) and t >= 0})

    class _LocalChat:
        @staticmethod
        def create(model, messages, max_tokens=MAX_GEN_TOKEN,
                   temperature=0.1, **kw):
            # 대화틀(chat template)로 messages를 모델 입력 형식으로 인코딩
            inputs = _local_tokenizer.apply_chat_template(
                messages, add_generation_prompt=True,
                return_tensors='pt', return_dict=True,
            ).to(device)
            with torch.no_grad():
                output_ids = _local_model.generate(
                    **inputs,
                    max_new_tokens=max_tokens,
                    do_sample=False,            # 일관된 결과를 위해 그리디 디코딩
                    eos_token_id=_terminators,
                    pad_token_id=_local_tokenizer.eos_token_id,
                )
            # 새로 생성된 토큰만 디코딩(입력 프롬프트 부분 제외)
            input_len = inputs['input_ids'].shape[-1]
            generated = output_ids[0][input_len:]
            text = _local_tokenizer.decode(generated, skip_special_tokens=True)
            return SimpleNamespace(
                choices=[SimpleNamespace(
                    message=SimpleNamespace(content=text.strip()))])

    class _LocalClient:
        def __init__(self):
            self.chat = SimpleNamespace(completions=_LocalChat())

    groq_client = _LocalClient()

## [코드 12-20] Groq 클라이언트 생성과 호출

`chat.completions.create()`는 JSON 응답을 파이썬 객체(`ChatCompletion`)로 변환해
반환하며, `choices[0].message.content`에서 본문을 꺼낸다. 본문 [코드 12-20]은
키를 직접 지정하지만(가독성 우선), 노트북은 환경 변수로 키를 읽거나 키가 없으면
로컬 LLM으로 안전하게 대체한다.

In [10]:
# 코드 12-20 - Groq 클라이언트 생성과 호출
response = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[{'role': 'user', 'content': '트랜스포머 모델은 어떤 구조인가요?'}],
    max_tokens=MAX_GEN_TOKEN,
    temperature=0.1,                 # 일관성을 위해 낮은 온도
)
print('생성 결과:\n' + response.choices[0].message.content)


생성 결과:
트랜스포머는 어텐션 기반 신경망 구조로 입력 시퀀스를 동시에 처리하며, 대부분의 대형 언어 모델의 기본 구조입니다.


## [코드 12-21] RAG 프롬프트 구성과 생성 함수

시스템 메시지에는 어조·안전 가이드라인을, 사용자 메시지에는 검색된 문서와 그
문서를 참고해 답하라는 지시를 담는다. 맨 앞의 "문서에 없는 내용은 추측하지 말라"는
한 줄이 RAG의 핵심 제어 장치다.


In [11]:
# 코드 12-21 - RAG 프롬프트 구성과 생성 함수
def generate_with_groq(prompt, model=GROQ_MODEL, max_tokens=MAX_GEN_TOKEN):
    response = groq_client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system',
             'content': '전문성 있는 한국어 문장으로 답변하며, 자료가 제공되지 '
                        '않은 사항은 결코 추측해 답하지 말고 모른다고 응답한다.'},
            {'role': 'user', 'content': prompt},
        ],
        max_tokens=max_tokens,
        temperature=0.1,
    )
    return response.choices[0].message.content


def build_rag_prompt(query, retrieved_docs):
    # 검색된 문서를 컨텍스트로 결합
    context = '\n\n'.join([
        f"[문서 {i + 1}: {doc['title']}]\n{doc['content']}"
        for i, doc in enumerate(retrieved_docs)
    ])
    # 컨텍스트와 질문을 한 프롬프트로 묶고, 답변 지침을 맨 앞에 둔다.
    prompt = (
        '다음 문서에 없는 내용은 결코 추측해 답하지 말고 모른다고 답하며, '
        '문서를 참고해 질문에 답한다.\n\n'
        f'* 참고 문서\n{context}\n\n'
        f'* 질문\n{query}'
    )
    return prompt


## [코드 12-22] 통합 RAG 파이프라인

검색 파이프라인과 생성 파이프라인을 한 함수로 묶는다. `generate_fn`을 인자로
받아 생성 함수만 교체할 수 있게 한다.


In [12]:
# 코드 12-22 - 통합 RAG 파이프라인
def rag_pipeline(query, documents, doc_embeddings, embed_model,
                 generate_fn, top_k=2):
    # 1단계: 검색
    retrieved_docs, scores = retrieve(
        query, documents, doc_embeddings, embed_model, top_k=top_k,
    )
    # 2단계: 프롬프트 구성
    prompt = build_rag_prompt(query, retrieved_docs)
    # 3단계: 생성
    answer = generate_fn(prompt)
    return answer, retrieved_docs, scores


### 참고 - RAG 적용/미적용 답변 비교 ([표 12-12] 대응)

성격이 다른 세 질문((1) 최신 회사 정보, (2) 환각 유도, (3) 문서 범위 밖)으로
RAG 적용/미적용 답변을 비교한다. Groq(8B)를 사용하면 본문 [표 12-12]에 가까운
결과가 나온다. 로컬 폴백(Bllossom-3B)은 모델이 작아 문서 범위 밖 질문을 또렷이
거절하지 못하는 등 답변이 다를 수 있다.

In [13]:
# 참고 - RAG vs 비-RAG 답변 비교 ([표 12-12] 대응)
test_queries = [
    'Groq는 어떤 회사이고, 어떤 모델을 API로 제공하나요?',       # 최신 정보
    'RAG는 누가 언제 발표했나요? 파이프라인은 어떻게 구성되나요?',  # 환각 유도
    '파이토치와 텐서플로의 차이점은 무엇인가요?',                # 문서 범위 밖
]

for q in test_queries:
    print('=' * 60)
    print(f'질문: {q}\n')
    print('--- RAG 없이 (LLM 단독) ---')
    print(generate_with_groq(q))
    print('\n--- RAG 적용 ---')
    answer, docs, rag_scores = rag_pipeline(
        q, documents, doc_embeddings, embed_model, generate_with_groq,
    )
    titles = [d['title'] for d in docs]
    print(f'(검색된 문서: {titles}, '
          f'유사도: {[round(s, 3) for s in rag_scores]})')
    print(answer, '\n')


질문: Groq는 어떤 회사이고, 어떤 모델을 API로 제공하나요?

--- RAG 없이 (LLM 단독) ---
Groq는 미국의 기술 회사로 알려져 있으나, 제공 모델 등 정확한 최신 정보는 제공하기 어렵습니다.

--- RAG 적용 ---
(검색된 문서: ['Groq', 'GPT 시리즈'], 유사도: [0.477, 0.46])
Groq는 LPU(언어 처리 장치)라는 AI 추론 전용 칩을 개발한 스타트업이며, 오픈소스 LLM의 API 서비스를 OpenAI 호환 인터페이스로 제공합니다. 

질문: RAG는 누가 언제 발표했나요? 파이프라인은 어떻게 구성되나요?

--- RAG 없이 (LLM 단독) ---
RAG는 Rapid Automated Glucose Monitoring의 약자로 추정되는 의료 모니터링 기술로 보입니다.

--- RAG 적용 ---
(검색된 문서: ['GPT 시리즈', 'RAG(검색 증강 생성)'], 유사도: [0.422, 0.345])
RAG는 2020년 메타 AI 연구팀이 발표했으며, 임베딩→검색→생성 3단계로 구성됩니다. 

질문: 파이토치와 텐서플로의 차이점은 무엇인가요?

--- RAG 없이 (LLM 단독) ---
파이토치는 동적 계산 그래프, 텐서플로는 정적 계산 그래프(1.x)가 대표적 차이이며, 최근에는 사용성이 비슷해졌습니다.

--- RAG 적용 ---
(검색된 문서: ['RAG(검색 증강 생성)', 'GPT 시리즈'], 유사도: [0.43, 0.413])
제공된 문서에 관련 내용이 없어 답할 수 없습니다. 

